In [ ]:
from pathlib import Path
import json

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path.cwd().parent.resolve()  # Ejecutar desde pec3/notebooks

SMART_DIR = BASE_DIR / "data" / "SMART-OM"

OUTPUT_DIR = BASE_DIR / "outputs"
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("SMART_DIR existe:", SMART_DIR.exists())
print("SMART_DIR:", SMART_DIR)

In [ ]:
json_files = list(SMART_DIR.rglob("*.json"))

json_region = []

for f in json_files:
    ruta = str(f).lower()
    
    if "region annotation" in ruta:
        json_region.append(f)

print("JSON totales:", len(json_files))
print("JSON de region annotation:", len(json_region))

for f in json_region[:10]:
    print(f.relative_to(SMART_DIR))

In [ ]:
def leer_json(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    return data

In [ ]:
def extraer_anotaciones_via(data, json_path):
    """
    Lee un JSON de VIA y devuelve una lista.
    
    Cada elemento tiene:
    - json_path
    - filename
    - regions
    """
    
    resultados = []
    
    if "_via_img_metadata" not in data:
        return resultados
    
    metadata = data["_via_img_metadata"]
    
    for img_id in metadata:
        item = metadata[img_id]
        
        filename = item.get("filename", None)
        regions = item.get("regions", [])
        
        if filename is None:
            continue
        
        # En VIA, regions a veces es lista y a veces diccionario
        if isinstance(regions, dict):
            regions = list(regions.values())
        
        resultados.append({
            "json_path": json_path,
            "filename": filename,
            "regions": regions
        })
    
    return resultados

In [ ]:
def limpiar_regions(regions):
    """
    Limpia regions para evitar None u objetos raros.
    Devuelve siempre una lista de diccionarios válidos.
    """
    
    if regions is None:
        return []
    
    if isinstance(regions, dict):
        regions = list(regions.values())
    
    if not isinstance(regions, list):
        return []
    
    regions_limpias = []
    
    for region in regions:
        if region is None:
            continue
        
        if not isinstance(region, dict):
            continue
        
        if "shape_attributes" not in region:
            continue
        
        regions_limpias.append(region)
    
    return regions_limpias


def contar_regiones_utiles(regions):
    """
    Cuenta regiones que se pueden convertir a máscara.
    """
    
    regions = limpiar_regions(regions)
    
    total = 0
    
    for region in regions:
        shape = region.get("shape_attributes", {})
        nombre = shape.get("name", "")
        
        if nombre in ["polygon", "rect", "circle", "ellipse"]:
            total += 1
    
    return total

In [ ]:
def buscar_imagen(json_path, filename):
    """
    Busca la imagen asociada a una anotación.
    Primero busca cerca del JSON y luego en todo SMART-OM.
    """
    
    if filename is None:
        return None
    
    filename = Path(str(filename)).name
    
    extensiones = [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]
    
    # Carpetas cercanas al JSON
    carpetas_busqueda = []
    
    carpetas_busqueda.append(json_path.parent)
    
    if json_path.parent.parent.exists():
        carpetas_busqueda.append(json_path.parent.parent)
    
    if json_path.parent.parent.parent.exists():
        carpetas_busqueda.append(json_path.parent.parent.parent)
    
    # Buscar por nombre exacto cerca
    for carpeta in carpetas_busqueda:
        posibles = list(carpeta.rglob(filename))
        
        for p in posibles:
            if p.suffix in extensiones:
                return p
    
    # Buscar por nombre exacto en todo SMART-OM
    posibles = list(SMART_DIR.rglob(filename))
    
    for p in posibles:
        if p.suffix in extensiones:
            return p
    
    # Buscar por stem
    stem = Path(filename).stem
    
    for ext in extensiones:
        posibles = list(SMART_DIR.rglob(stem + ext))
        
        if len(posibles) > 0:
            return posibles[0]
    
    return None

In [ ]:
json_path = json_region[0]
data = leer_json(json_path)

print("JSON elegido:")
print(json_path.relative_to(SMART_DIR))

print("\nClaves principales:")
print(list(data.keys()))

anotaciones = extraer_anotaciones_via(data, json_path)

print("\nAnotaciones extraídas:", len(anotaciones))

for a in anotaciones[:5]:
    print("filename:", a["filename"])
    print("num regions:", len(a["regions"]))
    print("regiones útiles:", contar_regiones_utiles(a["regions"]))
    print("------")

In [ ]:
entrada_valida = None
imagen_valida = None

for jp in json_region:
    data_tmp = leer_json(jp)
    anotaciones_tmp = extraer_anotaciones_via(data_tmp, jp)
    
    for a in anotaciones_tmp:
        n_regiones = contar_regiones_utiles(a["regions"])
        
        if n_regiones == 0:
            continue
        
        img_path = buscar_imagen(jp, a["filename"])
        
        if img_path is None:
            continue
        
        img_test = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
        
        if img_test is None:
            continue
        
        entrada_valida = a
        imagen_valida = img_path
        break
    
    if entrada_valida is not None:
        break

print("Entrada válida encontrada:")

if entrada_valida is None:
    print("No se ha encontrado ninguna entrada válida.")
else:
    print("JSON:", entrada_valida["json_path"].relative_to(SMART_DIR))
    print("Imagen:", entrada_valida["filename"])
    print("Ruta imagen:", imagen_valida)
    print("Regiones:", len(entrada_valida["regions"]))
    print("Regiones útiles:", contar_regiones_utiles(entrada_valida["regions"]))

In [ ]:
def crear_mascara_desde_regions(regions, alto, ancho):
    """
    Crea una máscara binaria a partir de regiones VIA.
    Fondo = 0
    Región = 255
    """
    
    mask = np.zeros((alto, ancho), dtype=np.uint8)
    
    regions = limpiar_regions(regions)
    
    for region in regions:
        shape = region.get("shape_attributes", {})
        nombre = shape.get("name", "")
        
        if nombre == "polygon":
            xs = shape.get("all_points_x", [])
            ys = shape.get("all_points_y", [])
            
            if len(xs) >= 3 and len(xs) == len(ys):
                puntos = np.array(list(zip(xs, ys)), dtype=np.int32)
                puntos = puntos.reshape((-1, 1, 2))
                cv2.fillPoly(mask, [puntos], 255)
        
        elif nombre == "rect":
            x = int(shape.get("x", 0))
            y = int(shape.get("y", 0))
            w = int(shape.get("width", 0))
            h = int(shape.get("height", 0))
            
            if w > 0 and h > 0:
                cv2.rectangle(mask, (x, y), (x + w, y + h), 255, -1)
        
        elif nombre == "circle":
            cx = int(shape.get("cx", 0))
            cy = int(shape.get("cy", 0))
            r = int(shape.get("r", 0))
            
            if r > 0:
                cv2.circle(mask, (cx, cy), r, 255, -1)
        
        elif nombre == "ellipse":
            cx = int(shape.get("cx", 0))
            cy = int(shape.get("cy", 0))
            rx = int(shape.get("rx", 0))
            ry = int(shape.get("ry", 0))
            
            if rx > 0 and ry > 0:
                cv2.ellipse(mask, (cx, cy), (rx, ry), 0, 0, 360, 255, -1)
    
    return mask

In [ ]:
if entrada_valida is None:
    raise ValueError("No hay entrada válida. Revisa la celda anterior.")

img = cv2.imread(str(imagen_valida), cv2.IMREAD_COLOR)

if img is None:
    raise ValueError("OpenCV no pudo leer la imagen")

alto, ancho = img.shape[:2]

mask = crear_mascara_desde_regions(
    entrada_valida["regions"],
    alto,
    ancho
)

img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12, 5))

plt.subplot(1, 3, 1)
plt.imshow(img_rgb)
plt.title("Imagen original")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(mask, cmap="gray")
plt.title("Máscara")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(img_rgb)
plt.imshow(mask, cmap="jet", alpha=0.35)
plt.title("Imagen + máscara")
plt.axis("off")

plt.tight_layout()

fig_path = FIGURES_DIR / "smart_om_primera_mascara.png"
plt.savefig(fig_path, dpi=150)
plt.show()

print("Figura guardada:", fig_path)

In [ ]:
filas = []

for jp in json_region:
    data_tmp = leer_json(jp)
    anotaciones_tmp = extraer_anotaciones_via(data_tmp, jp)
    
    for a in anotaciones_tmp:
        n_regiones = contar_regiones_utiles(a["regions"])
        
        if n_regiones == 0:
            continue
        
        img_path = buscar_imagen(jp, a["filename"])
        
        if img_path is None:
            continue
        
        filas.append({
            "json_path": str(jp),
            "json_relativo": str(jp.relative_to(SMART_DIR)),
            "filename": a["filename"],
            "img_path": str(img_path),
            "img_relativa": str(img_path.relative_to(SMART_DIR)),
            "num_regions": len(a["regions"]),
            "num_regions_utiles": n_regiones
        })

df_segmentacion = pd.DataFrame(filas)

print("Anotaciones utilizables:", len(df_segmentacion))
display(df_segmentacion.head())

csv_path = TABLES_DIR / "smart_om_region_annotations_validas.csv"
df_segmentacion.to_csv(csv_path, index=False)

print("CSV guardado en:", csv_path)

In [ ]:
muestras = df_segmentacion.sample(
    n=min(6, len(df_segmentacion)),
    random_state=7
)

plt.figure(figsize=(12, 8))

for i, (_, fila) in enumerate(muestras.iterrows()):
    jp = Path(fila["json_path"])
    img_path = Path(fila["img_path"])
    
    data_tmp = leer_json(jp)
    anotaciones_tmp = extraer_anotaciones_via(data_tmp, jp)
    
    entrada = None
    
    for a in anotaciones_tmp:
        if a["filename"] == fila["filename"]:
            entrada = a
            break
    
    if entrada is None:
        continue
    
    img = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
    
    if img is None:
        continue
    
    alto, ancho = img.shape[:2]
    
    mask = crear_mascara_desde_regions(
        entrada["regions"],
        alto,
        ancho
    )
    
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    plt.subplot(2, 3, i + 1)
    plt.imshow(img_rgb)
    plt.imshow(mask, cmap="jet", alpha=0.35)
    plt.title(f"regions={fila['num_regions_utiles']}")
    plt.axis("off")

plt.tight_layout()

fig_path = FIGURES_DIR / "smart_om_muestras_mascaras.png"
plt.savefig(fig_path, dpi=150)
plt.show()

print("Figura guardada:", fig_path)